# Benchmark: `allc → mcds → h5ad` (ALLCools / pym3c) vs `cz → h5ad` (cytozip)

Compare the two routes from single-cell methylation calls to an `AnnData` (`.h5ad`) feature matrix
on the mouse developing frontal cortex snm3C-seq data set (10 age groups, ~6k cells each, P56 ~21.6k cells):

| | Pipeline A — **ALLCools / pym3c** | Pipeline B — **cytozip** |
|---|---|---|
| input | `mapped/<age>/allc/*.allc.tsv.gz` (tabix) | `mapped/<age>/cz/*.cz` (from `czip allc2cz`, reference = `mm10_with_chrL.allc.cz`) |
| step 1 | `allcools generate-dataset` → `<age>.100kb.mcds` (`count CGN,CHN`) | `czip cz_to_anndata -f 100000 -s mm10_with_chrL.CGN.cz` → `100kb-CGN.h5ad` |
| step 2 | `allcools generate-dataset` → `<age>.gene.mcds` (`count CGN,CHN`) | `czip cz_to_anndata -f 100000 -s mm10_with_chrL.CHN.cz` → `100kb-CHN.h5ad` |
| step 3 | `pym3c adata generate_region_adata` → `100kb-CGN.h5ad`, `100kb-CHN.h5ad` | `czip cz_to_anndata -f gene_2000_2000.bed.gz -s mm10_with_chrL.CGN.cz` → `gene-CGN.h5ad` |
| step 4/5 | `pym3c adata generate_gadata` ×2 → `gene-CGN.h5ad`, `gene-CHN.h5ad` | `czip cz_to_anndata -f gene_2000_2000.bed.gz -s mm10_with_chrL.CHN.cz` → `gene-CHN.h5ad` |
| output | 4 × h5ad, `.X` = mC fraction | 4 × h5ad, `.X` = mC fraction, `.layers['mc'|'cov']` raw counts |

Both pipelines run **sequentially on the same node with identical resources** (`NCPU` worker processes each,
96 CPUs / 220 GB node) and are driven by the same commands pym3c uses in production
(`pym3c/notebooks/projects/mouse_pfc.md`). Every stage is wrapped by `scripts/bench_run.py`, which samples the
whole process tree once per second and records:

* `wall_s` — wall-clock time; `cpu_user_s` / `cpu_sys_s` — exact CPU time from `wait4()` (all workers included)
* `max_rss_sum_mb` — peak Σ RSS over all processes (what `top` / SLURM `MaxRSS`-like accounting shows)
* `max_pss_sum_mb` — peak Σ PSS (proportional set size: pages shared between fork workers counted once → *true* physical footprint)
* `max_rss_single_mb` — largest single-process RSS (what `/usr/bin/time %M` reports)
* `output_size_mb` — size of the produced file(s)

Everything lives in `~/Projects/test_cytozip/to_adata/` (scripts, logs, mcds, h5ad, tables, figures).
A second, very different dataset — **188 human basal-ganglia cluster pseudobulks (hg38, ~6 GB allc each, 1.1 TB)** —
is benchmarked with the same scripts (`DATASET=bg_cluster`, results in `to_adata/bg_cluster/`); see section 11.
The one-off `allc → cz` conversion (`czip allc2cz`) is timed as well so it can optionally be charged to pipeline B.


## 0. How to run (one interactive node, sequential)

All steps run **one after another on a single node** (96 CPUs / 220 GB). Both pipelines use the same
`NCPU` (default 96, `env.sh`); the one-off `allc → cz` uses `A2C_NCPU` (default 96).

```shell
salloc -A mcb130189 -p shared -N 1 -n 96 --mem=220G -t 2-00:00:00     # or sinteractive
cd ~/Projects/test_cytozip/to_adata
tmux                                   # recommended: survives a dropped SSH session
bash run_all.sh                        # 00 -> 01 -> 02 -> 03 -> 04 -> 05, log in logs/run_all.<date>.log
# NCPU=48 bash run_all.sh              # fewer workers for the two pipelines (always identical for both)
# STEPS="02 03" bash run_all.sh        # only some steps
```

or step by step:

```shell
bash scripts/00_prepare.sh             # check the pre-built CGN/CHN indexes, write tables/<age>.allc_path.tsv, mkdir mapped/<age>/cz
bash scripts/01_allc2cz.sh             # allc -> cz for every age (+ timing / size logs)
bash scripts/02_allcools_pipeline.sh   # pipeline A: allc -> mcds -> h5ad for every age
bash scripts/03_cytozip_pipeline.sh    # pipeline B: cz -> h5ad for every age (requires 01 done for that age)
bash scripts/04_make_subsets.sh        # nested random P56 subsets (500 ... 16000 cells)
bash scripts/05_scaling.sh             # both pipelines on every subset
# any script also accepts sample names: bash scripts/02_allcools_pipeline.sh P0 P56
```

**Resumable:** every stage is wrapped by `bench_run.py --resume`; a stage whose `logs/<pipeline>.<name>.tsv`
already has an `exit_code == 0` row is skipped, and stale outputs of a failed / interrupted stage are deleted
before it is redone. If the allocation dies, simply re-run the same command. `FORCE=1` redoes finished work
(`01_allc2cz.sh` additionally refuses to run on a half-filled `cz/` dir without `FORCE=1`, because
`czip allc2cz` skips existing files and the time would be wrong).

**Second dataset (human BG pseudobulk, hg38).** Same scripts, selected with `DATASET=bg_cluster`
(overrides in `datasets/bg_cluster.sh`; results in `to_adata/bg_cluster/`, cz written to
`~/Projects/BG/pseudobulk/Cluster/cz/`):

```shell
DATASET=bg_cluster bash run_all.sh             # 00 -> 01 -> 02 -> 03 (04/05 are skipped: no scaling series)
# or step by step: DATASET=bg_cluster bash scripts/01_allc2cz.sh   (etc.)
```

Resulting layout:

```
to_adata/
├── env.sh                      shared config (paths, NCPU, A2C_NCPU, samples, BENCH, stage_done(), DATASET switch)
├── datasets/bg_cluster.sh      hg38 / BG pseudobulk overrides (refs, -F H,H, A2C_NCPU=12, chunk_size 2)
├── run_all.sh                  runs steps 00-05 in order (STEPS=... to pick)
├── scripts/
│   ├── bench_run.py            process-tree time / memory monitor (+ --resume)  -> logs/<pipeline>.<name>.tsv
│   ├── 00_prepare.sh           check indexes, allc tables, cz dirs
│   ├── 01_allc2cz.sh           [samples...]                       -> mapped/<age>/cz/, logs/allc2cz.<age>.{tsv,size.tsv,cells.tsv}
│   ├── 02_allcools_pipeline.sh [samples...] | --one <name> <allc_table>   -> mcds/, allcools/<name>/*.h5ad
│   ├── 03_cytozip_pipeline.sh  [samples...] | --one <name> <cz_dir>       -> cytozip/<name>/*.h5ad
│   ├── 04_make_subsets.sh      P56 subsets -> tables/P56_sub*.allc_path.tsv, cz_subsets/P56_sub*/
│   └── 05_scaling.sh           [sizes...]  runs 02/03 --one for every subset
├── logs/  tables/  figures/  mcds/  allcools/  cytozip/  cz_subsets/      (mouse)
├── bg_cluster/{logs,tables,figures,mcds,allcools,cytozip}/            (human BG pseudobulk)
```


In [ ]:
import os, sys, glob, re, textwrap, subprocess
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

WORKDIR = os.path.expanduser("~/Projects/test_cytozip/to_adata")
os.chdir(WORKDIR)
LOG_DIR, TABLE_DIR, FIG_DIR = "logs", "tables", "figures"
os.makedirs(TABLE_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

SAMPLES = ['E15', 'E17', 'P0', 'P3', 'P6', 'P9', 'P12', 'P13', 'P28', 'P56']
STAGES = ['100kb-CGN', '100kb-CHN', 'gene-CGN', 'gene-CHN']

# manuscript plotting style (same as notebooks/benchmark/benchmark_methylation_calling.ipynb)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 8
mpl.rcParams['axes.linewidth'] = 0.6
COLORS = {'ALLCools': '#1f77b4', 'cytozip': '#d7263d', 'cytozip (+allc2cz)': '#f4a261'}
LABEL = {'allcools': 'ALLCools', 'cytozip': 'cytozip'}

def savefig(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f"{FIG_DIR}/{name}.{ext}", bbox_inches='tight', dpi=300)
    print('saved', f"{FIG_DIR}/{name}.pdf/.png")

def despine(ax):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

print(subprocess.run(['bash', '-c', 'source env.sh; echo NCPU=$NCPU A2C_NCPU=$A2C_NCPU; czip --version 2>/dev/null || python -c "import cytozip;print(cytozip.__version__)"; allcools --version'],
                     capture_output=True, text=True).stdout)

### Scripts used (for the record)

In [ ]:
for f in ['env.sh', 'datasets/bg_cluster.sh', 'run_all.sh', 'scripts/bench_run.py', 'scripts/00_prepare.sh', 'scripts/01_allc2cz.sh',
          'scripts/02_allcools_pipeline.sh', 'scripts/03_cytozip_pipeline.sh', 'scripts/04_make_subsets.sh', 'scripts/05_scaling.sh']:
    print('=' * 100); print('#', f); print('=' * 100)
    print(open(f).read())

## 1. Progress: which outputs exist?

In [ ]:
rows = []
names = SAMPLES + sorted({re.sub(r'^.*?\.(P56_sub\d+)\.tsv$', r'\1', p) for p in glob.glob(f'{LOG_DIR}/*.P56_sub*.tsv')},
                         key=lambda x: int(x.split('sub')[1]))
for s in names:
    r = {'name': s}
    r['cz_files'] = len(glob.glob(os.path.expanduser(f'~/Projects/mouse_dev/mapped/{s}/cz/*.cz'))) if s in SAMPLES else len(glob.glob(f'cz_subsets/{s}/*.cz'))
    r['mcds'] = sum(os.path.exists(f'mcds/{s}.{d}.mcds') for d in ('100kb', 'gene'))
    for p in ('allcools', 'cytozip'):
        r[f'{p}_h5ad'] = sum(os.path.exists(f'{p}/{s}/{s}.{st}.h5ad') for st in STAGES)
        r[f'{p}_log'] = os.path.exists(f'{LOG_DIR}/{p}.{s}.tsv')
    rows.append(r)
progress = pd.DataFrame(rows).set_index('name')
progress

## 2. Collect the per-stage logs

`logs/<pipeline>.<name>.tsv` — one row per stage written by `bench_run.py`. Stage labels are harmonised to
`mcds-100kb`, `mcds-gene` (pipeline A only) and `h5ad-100kb-CGN`, `h5ad-100kb-CHN`, `h5ad-gene-CGN`, `h5ad-gene-CHN`
(pipeline A writes `100kb-CGN` and `100kb-CHN` in a single `generate_region_adata` call → stage `h5ad-100kb-CGN+CHN`).

In [ ]:
logs = []
for f in sorted(glob.glob(f'{LOG_DIR}/*.tsv')):
    if f.endswith(('.size.tsv', '.cells.tsv')):   # size tables, not bench_run.py logs
        continue
    d = pd.read_csv(f, sep='\t')
    if 'pipeline' not in d.columns or 'wall_s' not in d.columns:
        continue
    d['log_file'] = os.path.basename(f)
    logs.append(d)
log = pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()
if len(log):
    # keep the last successful run of every (pipeline, name, stage)
    log = log.sort_values('start_time').drop_duplicates(['pipeline', 'name', 'stage'], keep='last')
    log['exit_ok'] = log['exit_code'] == 0
    log['cpu_s'] = log['cpu_user_s'] + log['cpu_sys_s']
    log['wall_min'] = log['wall_s'] / 60
    log['max_pss_sum_gb'] = log['max_pss_sum_mb'] / 1024
    log['max_rss_sum_gb'] = log['max_rss_sum_mb'] / 1024
    log['is_subset'] = log['name'].str.contains('_sub')
    log.to_csv(f'{TABLE_DIR}/all_stages.tsv', sep='\t', index=False)
    bad = log.loc[~log.exit_ok, ['pipeline', 'name', 'stage', 'exit_code', 'log_file']]
    if len(bad):
        print('WARNING — failed stages (excluded below):'); display(bad)
    log = log[log.exit_ok]
    display(log.groupby(['pipeline', 'name']).size().unstack(0))
else:
    print('no logs yet — run the SLURM jobs first')
log.head()

## 3. Per-sample summary

For every (pipeline, sample): total wall time = Σ stages, total CPU time = Σ stages, peak memory = max over stages
(`max_pss_sum` is the fair physical footprint; `max_rss_sum` is also reported), output size = Σ h5ad sizes
(+ the intermediate `.mcds` for pipeline A). `cytozip (+allc2cz)` additionally charges the one-off `allc → cz`
conversion of that sample to pipeline B.

In [ ]:
def summarise(df):
    g = df.groupby(['pipeline', 'name'])
    s = pd.DataFrame({
        'n_cells': g['n_cells'].max(),
        'ncpu': g['ncpu'].max(),
        'n_stages': g.size(),
        'wall_s': g['wall_s'].sum(),
        'cpu_s': g['cpu_s'].sum(),
        'peak_pss_mb': g['max_pss_sum_mb'].max(),
        'peak_rss_mb': g['max_rss_sum_mb'].max(),
        'peak_rss_single_mb': g['max_rss_single_mb'].max(),
        'output_mb': g['output_size_mb'].sum(),
    }).reset_index()
    s['wall_min'] = s['wall_s'] / 60
    s['cpu_h'] = s['cpu_s'] / 3600
    s['peak_pss_gb'] = s['peak_pss_mb'] / 1024
    s['peak_rss_gb'] = s['peak_rss_mb'] / 1024
    return s

main = log[log.pipeline.isin(['allcools', 'cytozip'])]
summary = summarise(main)
# h5ad-only output size (exclude intermediate mcds) for a like-for-like file-size comparison
h5ad_only = main[main.stage.str.startswith('h5ad')].groupby(['pipeline', 'name'])['output_size_mb'].sum().rename('h5ad_mb').reset_index()
summary = summary.merge(h5ad_only, on=['pipeline', 'name'], how='left')

# cytozip + allc2cz (one-off conversion charged to pipeline B)
a2c = log[log.pipeline == 'allc2cz'].set_index('name')
if len(a2c):
    cz_plus = summary[summary.pipeline == 'cytozip'].copy()
    cz_plus['pipeline'] = 'cytozip+allc2cz'
    base = cz_plus['name'].str.replace(r'_sub\d+$', '', regex=True)  # subsets share the parent's conversion
    frac = np.where(cz_plus['name'].str.contains('_sub'), cz_plus['n_cells'] / base.map(a2c['n_cells']), 1.0)
    cz_plus['wall_s'] += base.map(a2c['wall_s']).values * frac
    cz_plus['cpu_s'] += base.map(a2c['cpu_s']).values * frac
    cz_plus['peak_pss_mb'] = np.maximum(cz_plus['peak_pss_mb'], base.map(a2c['max_pss_sum_mb']).values)
    cz_plus['peak_rss_mb'] = np.maximum(cz_plus['peak_rss_mb'], base.map(a2c['max_rss_sum_mb']).values)
    cz_plus['wall_min'] = cz_plus['wall_s'] / 60; cz_plus['cpu_h'] = cz_plus['cpu_s'] / 3600; cz_plus['peak_pss_gb'] = cz_plus['peak_pss_mb'] / 1024; cz_plus['peak_rss_gb'] = cz_plus['peak_rss_mb'] / 1024
    summary = pd.concat([summary, cz_plus], ignore_index=True)

summary['is_subset'] = summary['name'].str.contains('_sub')
summary.to_csv(f'{TABLE_DIR}/summary_per_sample.tsv', sep='\t', index=False)
full = summary[~summary.is_subset]
display(full.pivot(index='name', columns='pipeline', values=['n_cells', 'wall_min', 'cpu_h', 'peak_pss_gb', 'peak_rss_gb', 'h5ad_mb']).reindex(SAMPLES).round(2))

In [ ]:
# speed-up and memory ratio per sample (ALLCools / cytozip)
piv = full.pivot(index='name', columns='pipeline')
ratio = pd.DataFrame({
    'n_cells': piv['n_cells']['cytozip'],
    'speedup_wall (ALLCools/cytozip)': piv['wall_s']['allcools'] / piv['wall_s']['cytozip'],
    'speedup_wall (ALLCools/cytozip+allc2cz)': piv['wall_s']['allcools'] / piv['wall_s'].get('cytozip+allc2cz', np.nan),
    'speedup_cpu (ALLCools/cytozip)': piv['cpu_s']['allcools'] / piv['cpu_s']['cytozip'],
    'mem_ratio_pss (cytozip/ALLCools)': piv['peak_pss_mb']['cytozip'] / piv['peak_pss_mb']['allcools'],
    'mem_ratio_rss (cytozip/ALLCools)': piv['peak_rss_mb']['cytozip'] / piv['peak_rss_mb']['allcools'],
    'h5ad_size_ratio (cytozip/ALLCools)': piv['h5ad_mb']['cytozip'] / piv['h5ad_mb']['allcools'],
}).reindex(SAMPLES)
ratio.loc['mean'] = ratio.mean(numeric_only=True)
ratio.to_csv(f'{TABLE_DIR}/speedup_per_sample.tsv', sep='\t')
display(ratio.round(2))

tot = full.groupby('pipeline')[['wall_s', 'cpu_s']].sum()
print(textwrap.dedent(f'''
    === totals over {full.name.nunique()} samples, {int(full.groupby("pipeline").n_cells.sum().max()):,} cells ===
    ALLCools          wall = {tot.loc["allcools","wall_s"]/3600:6.2f} h   cpu = {tot.loc["allcools","cpu_s"]/3600:7.1f} CPU-h
    cytozip           wall = {tot.loc["cytozip","wall_s"]/3600:6.2f} h   cpu = {tot.loc["cytozip","cpu_s"]/3600:7.1f} CPU-h
    speed-up (wall)   = {tot.loc["allcools","wall_s"]/tot.loc["cytozip","wall_s"]:.2f}x
    speed-up (cpu)    = {tot.loc["allcools","cpu_s"]/tot.loc["cytozip","cpu_s"]:.2f}x
    peak PSS (max over samples)  ALLCools = {full[full.pipeline=="allcools"].peak_pss_gb.max():.1f} GB   cytozip = {full[full.pipeline=="cytozip"].peak_pss_gb.max():.1f} GB
'''))

## 4. Figure 1 — overall comparison (mean ± SD over the 10 age groups)

Panels: total wall time, total CPU time, peak memory (Σ PSS), h5ad output size.

In [ ]:
order = ['allcools', 'cytozip'] + (['cytozip+allc2cz'] if 'cytozip+allc2cz' in full.pipeline.unique() else [])
labels = {'allcools': 'ALLCools', 'cytozip': 'cytozip', 'cytozip+allc2cz': 'cytozip\n(+allc2cz)'}
cols = {'allcools': COLORS['ALLCools'], 'cytozip': COLORS['cytozip'], 'cytozip+allc2cz': COLORS['cytozip (+allc2cz)']}
metrics = [('wall_min', 'Wall time (min)', 'Runtime'),
           ('cpu_h', 'CPU time (CPU-h)', 'CPU time'),
           ('peak_pss_gb', 'Peak memory (GB, Σ PSS)', 'Peak memory'),
           ('h5ad_mb', 'h5ad size (MB)', 'Output size')]

fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.4), constrained_layout=True)
for ax, (col, ylabel, title) in zip(axes, metrics):
    d = full.groupby('pipeline')[col].agg(['mean', 'std']).reindex(order)
    x = np.arange(len(order))
    ax.bar(x, d['mean'], yerr=d['std'], capsize=3, width=0.65, color=[cols[p] for p in order],
           error_kw=dict(ecolor='black', lw=0.8))
    # individual samples as dots
    for i, p in enumerate(order):
        v = full.loc[full.pipeline == p, col].values
        ax.scatter(np.random.default_rng(0).uniform(-0.15, 0.15, len(v)) + i, v, s=6, color='k', alpha=0.6, zorder=3, lw=0)
    ax.set_xticks(x); ax.set_xticklabels([labels[p] for p in order], fontsize=7)
    ax.set_ylabel(ylabel); ax.set_title(title, fontsize=9); despine(ax)
    if col in ('wall_min', 'cpu_h'):
        sp = d.loc['allcools', 'mean'] / d.loc['cytozip', 'mean']
        ax.text(0.98, 0.98, f'{sp:.1f}× faster', transform=ax.transAxes, ha='right', va='top', fontsize=7, color=COLORS['cytozip'])
    elif col == 'peak_pss_gb':
        r = d.loc['cytozip', 'mean'] / d.loc['allcools', 'mean']
        ax.text(0.98, 0.98, f'{r*100:.0f}% memory', transform=ax.transAxes, ha='right', va='top', fontsize=7, color=COLORS['cytozip'])
fig.suptitle(f'allc → mcds → h5ad (ALLCools) vs cz → h5ad (cytozip); n = {full.name.nunique()} samples, mean ± SD', fontsize=8)
savefig(fig, 'fig1_overall_mean_sd')
plt.show()

## 5. Figure 2 — stage breakdown (stacked wall time per pipeline)

In [ ]:
stage_order_a = ['mcds-100kb', 'mcds-gene', 'h5ad-100kb-CGN+CHN', 'h5ad-gene-CGN', 'h5ad-gene-CHN']
stage_order_b = ['h5ad-100kb-CGN', 'h5ad-100kb-CHN', 'h5ad-gene-CGN', 'h5ad-gene-CHN']
stage_mean = main[~main.is_subset].groupby(['pipeline', 'stage'])['wall_min'].mean().unstack(0)
stage_mean.to_csv(f'{TABLE_DIR}/stage_mean_wall_min.tsv', sep='\t')
display(stage_mean.round(2))

fig, ax = plt.subplots(figsize=(3.2, 2.6), constrained_layout=True)
cmap_a = plt.get_cmap('Blues')(np.linspace(0.45, 0.9, len(stage_order_a)))
cmap_b = plt.get_cmap('Reds')(np.linspace(0.45, 0.9, len(stage_order_b)))
for xi, (p, stages, cm) in enumerate([('allcools', stage_order_a, cmap_a), ('cytozip', stage_order_b, cmap_b)]):
    bottom = 0
    for st, c in zip(stages, cm):
        v = stage_mean[p].get(st, np.nan) if p in stage_mean else np.nan
        if np.isnan(v):
            continue
        ax.bar(xi, v, bottom=bottom, color=c, width=0.7, edgecolor='white', lw=0.4, label=f'{LABEL[p]}: {st}')
        bottom += v
    ax.text(xi, bottom, f'{bottom:.1f} min', ha='center', va='bottom', fontsize=7)
ax.set_xticks([0, 1]); ax.set_xticklabels(['ALLCools', 'cytozip'], fontsize=7)
ax.set_xlim(-0.6, 1.6)
ax.set_title('allc→mcds→h5ad  vs  cz→h5ad', fontsize=8)
ax.set_ylabel('Mean wall time per sample (min)'); despine(ax)
ax.legend(fontsize=6, frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left', title='stage', title_fontsize=6.5)
savefig(fig, 'fig2_stage_breakdown')
plt.show()

## 6. Figure 3 — per-sample runtime and peak memory

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.5), constrained_layout=True)
x = np.arange(len(SAMPLES)); w = 0.38
for ax, (col, ylabel) in zip(axes, [('wall_min', 'Wall time (min)'), ('peak_pss_gb', 'Peak memory (GB, Σ PSS)')]):
    for i, p in enumerate(['allcools', 'cytozip']):
        v = full[full.pipeline == p].set_index('name')[col].reindex(SAMPLES)
        ax.bar(x + (i - 0.5) * w, v, width=w, color=cols[p], label=LABEL[p])
    ax.set_xticks(x); ax.set_xticklabels(SAMPLES, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel(ylabel); despine(ax)
    ax2 = ax.twinx(); ax2.plot(x, full[full.pipeline == 'cytozip'].set_index('name')['n_cells'].reindex(SAMPLES) / 1000, 'k.--', lw=0.6, ms=3)
    ax2.set_ylabel('cells (×1000)', fontsize=7); ax2.spines['top'].set_visible(False)
axes[0].legend(frameon=False, fontsize=7)
savefig(fig, 'fig3_per_sample')
plt.show()

## 7. Figure 4 — scaling with the number of cells

Nested random subsets of P56 (500 … 16 000 cells) plus all full samples; log–log axes with a fitted power-law slope.

In [ ]:
sc = summary[summary.pipeline.isin(['allcools', 'cytozip'])].copy()
if sc.is_subset.any():
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4), constrained_layout=True)
    for ax, (col, ylabel) in zip(axes, [('wall_min', 'Wall time (min)'), ('cpu_h', 'CPU time (CPU-h)'), ('peak_pss_gb', 'Peak memory (GB, Σ PSS)')]):
        for p in ['allcools', 'cytozip']:
            d = sc[sc.pipeline == p].sort_values('n_cells')
            sub, fl = d[d.is_subset], d[~d.is_subset]
            ax.plot(sub.n_cells, sub[col], '-o', color=cols[p], ms=3.5, lw=1, label=f'{LABEL[p]} (P56 subsets)')
            ax.scatter(fl.n_cells, fl[col], marker='s', s=14, facecolor='none', edgecolor=cols[p], lw=0.8, label=f'{LABEL[p]} (full samples)')
            if len(sub) >= 3:
                k, b = np.polyfit(np.log10(sub.n_cells), np.log10(sub[col].clip(lower=1e-6)), 1)
                xs = np.array([sub.n_cells.min(), sub.n_cells.max()])
                ax.plot(xs, 10 ** (k * np.log10(xs) + b), ':', color=cols[p], lw=0.8)
                ax.text(0.02, 0.98 - 0.09 * (p == 'cytozip'), f'{LABEL[p]}: slope {k:.2f}', transform=ax.transAxes, fontsize=6.5, color=cols[p], va='top')
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('Number of cells'); ax.set_ylabel(ylabel); despine(ax)
    axes[-1].legend(fontsize=6, frameon=False, loc='lower right')
    savefig(fig, 'fig4_scaling_vs_cells')
    plt.show()
else:
    print('no scaling-series results yet (run scripts/04_make_subsets.sh then scripts/05_scaling.sh)')

## 8. `allc → cz` conversion: runtime, memory and file sizes

`scripts/01_allc2cz.sh <age>` runs `czip allc2cz -I allc/ -O cz/ -r mm10_with_chrL.allc.cz -j A2C_NCPU` (batch mode: the
reference is decoded once and shared with all workers by fork) under `bench_run.py`, so for every age group we have

* `logs/allc2cz.<age>.tsv` — wall / CPU time, peak Σ RSS / Σ PSS, largest single-process RSS, number of CPUs
* `logs/allc2cz.<age>.size.tsv` — total size of all `allc.tsv.gz` vs all `.cz`
* `logs/allc2cz.<age>.cells.tsv` — per-cell sizes (`allc.tsv.gz`, `.tbi`, `.cz`)


In [ ]:
size_files = glob.glob(f'{LOG_DIR}/allc2cz.*.size.tsv')
if size_files and len(a2c):
    sz = pd.concat([pd.read_csv(f, sep='\t') for f in size_files]).set_index('name').reindex(SAMPLES)
    sz['allc_gb'] = sz.allc_dir_mb / 1024; sz['cz_gb'] = sz.cz_dir_mb / 1024
    sz['cz/allc'] = sz.cz_dir_mb / sz.allc_dir_mb
    sz['allc_mb_per_cell'] = sz.allc_dir_mb / sz.n_allc; sz['cz_mb_per_cell'] = sz.cz_dir_mb / sz.n_cz
    sz['ncpu'] = a2c['ncpu'].reindex(SAMPLES)
    sz['wall_min'] = a2c['wall_s'].reindex(SAMPLES) / 60
    sz['cpu_h'] = a2c['cpu_s'].reindex(SAMPLES) / 3600
    sz['s_per_cell_wall'] = a2c['wall_s'].reindex(SAMPLES) / sz.n_allc
    sz['cpu_s_per_cell'] = a2c['cpu_s'].reindex(SAMPLES) / sz.n_allc
    sz['peak_pss_gb'] = a2c['max_pss_sum_mb'].reindex(SAMPLES) / 1024
    sz['peak_rss_gb'] = a2c['max_rss_sum_mb'].reindex(SAMPLES) / 1024
    sz['peak_rss_single_gb'] = a2c['max_rss_single_mb'].reindex(SAMPLES) / 1024
    sz['max_n_procs'] = a2c['max_n_procs'].reindex(SAMPLES)
    sz.loc['total/mean'] = sz.sum(numeric_only=True)
    for c in ['cz/allc', 'allc_mb_per_cell', 'cz_mb_per_cell', 'ncpu', 's_per_cell_wall', 'cpu_s_per_cell', 'peak_pss_gb', 'peak_rss_gb', 'peak_rss_single_gb', 'max_n_procs']:
        sz.loc['total/mean', c] = sz.loc[SAMPLES, c].mean() if c != 'cz/allc' else sz.loc[SAMPLES, 'cz_dir_mb'].sum() / sz.loc[SAMPLES, 'allc_dir_mb'].sum()
    sz.to_csv(f'{TABLE_DIR}/allc2cz_summary.tsv', sep='\t')
    display(sz[['n_allc', 'n_cz', 'ncpu', 'wall_min', 'cpu_h', 's_per_cell_wall', 'cpu_s_per_cell', 'peak_pss_gb', 'peak_rss_gb',
                'peak_rss_single_gb', 'allc_gb', 'cz_gb', 'cz/allc', 'allc_mb_per_cell', 'cz_mb_per_cell']].round(3))
    t = sz.loc['total/mean']
    print(f"allc2cz over {int(t.n_allc):,} cells: wall = {t.wall_min/60:.2f} h, cpu = {t.cpu_h:.1f} CPU-h, "
          f"mean peak PSS = {t.peak_pss_gb:.1f} GB; storage {t.allc_gb:.1f} GB allc -> {t.cz_gb:.1f} GB cz ({t['cz/allc']*100:.1f}%)")

    # per-cell sizes
    cell_files = glob.glob(f'{LOG_DIR}/allc2cz.*.cells.tsv')
    pc = pd.concat([pd.read_csv(f, sep='\t').assign(name=os.path.basename(f).split('.')[1]) for f in cell_files], ignore_index=True)
    pc = pc[pc.cz_bytes.notna()].copy()
    pc['allc_mb'] = pc.allc_bytes / 1024**2; pc['cz_mb'] = pc.cz_bytes.astype(float) / 1024**2; pc['ratio'] = pc.cz_bytes.astype(float) / pc.allc_bytes
    pc.to_csv(f'{TABLE_DIR}/allc2cz_per_cell_sizes.tsv.gz', sep='\t', index=False)
    print(f"per cell (n={len(pc):,}): allc median {pc.allc_mb.median():.1f} MB, cz median {pc.cz_mb.median():.1f} MB, "
          f"median cz/allc = {pc.ratio.median():.3f}")

    fig, axes = plt.subplots(1, 4, figsize=(7.4, 2.3), constrained_layout=True)
    x = np.arange(len(SAMPLES)); w = 0.38; s_ = sz.loc[SAMPLES]
    ax = axes[0]
    ax.bar(x, s_.wall_min, color=COLORS['cytozip'], width=0.65)
    ax.set_ylabel('Wall time (min)'); ax.set_title(f'allc→cz ({int(s_.ncpu.max())} CPUs)', fontsize=8)
    ax2 = ax.twinx(); ax2.plot(x, s_.n_allc / 1000, 'k.--', lw=0.6, ms=3); ax2.set_ylabel('cells (×1000)', fontsize=7); ax2.spines['top'].set_visible(False)
    ax = axes[1]
    ax.bar(x - w/2, s_.peak_pss_gb, w, color=COLORS['cytozip'], label='Σ PSS')
    ax.bar(x + w/2, s_.peak_rss_gb, w, color='#f4a261', label='Σ RSS')
    ax.set_ylabel('Peak memory (GB)'); ax.set_title('Peak memory', fontsize=8)
    ax.set_ylim(0, max(s_.peak_rss_gb.max(), s_.peak_pss_gb.max()) * 1.3); ax.legend(frameon=False, fontsize=6, loc='upper left', ncol=2)
    ax = axes[2]
    ax.bar(x - w/2, s_.allc_gb, w, color='#7f7f7f', label='allc.tsv.gz')
    ax.bar(x + w/2, s_.cz_gb, w, color=COLORS['cytozip'], label='.cz')
    ax.set_ylabel('Total size (GB)'); ax.set_title('Storage', fontsize=8)
    ax.set_ylim(0, s_.allc_gb.max() * 1.3); ax.legend(frameon=False, fontsize=6, loc='upper left')
    for ax in axes[:3]:
        ax.set_xticks(x); ax.set_xticklabels(SAMPLES, rotation=45, ha='right', fontsize=6.5); despine(ax)
    ax = axes[3]
    ax.hexbin(pc.allc_mb, pc.cz_mb, gridsize=50, bins='log', cmap='Reds', lw=0, mincnt=1)
    lim = max(pc.allc_mb.max(), pc.cz_mb.max())
    for r_ in (0.1, 0.2, 0.5):
        ax.plot([0, lim], [0, lim * r_], 'k:', lw=0.5); ax.text(lim, lim * r_, f'{r_:g}×', fontsize=5.5, va='bottom', ha='right')
    ax.set_xlabel('allc.tsv.gz size per cell (MB)'); ax.set_ylabel('.cz size per cell (MB)')
    ax.set_title(f'per cell (n={len(pc):,}), median cz/allc = {pc.ratio.median():.2f}', fontsize=7); despine(ax)
    savefig(fig, 'fig5_allc2cz_time_memory_storage')
    plt.show()
else:
    print('no allc2cz logs yet')

## 9. Validation — do both pipelines produce the same matrices?

For one sample, load the `100kb-CGN` / `gene-CGN` (and CHN) h5ad from both pipelines, align cells and features and compare
the **raw** mC fractions on entries covered in both: ALLCools/pym3c `.X` (`normalize=False` → `mc/cov`) versus
cytozip `layers['mc'] / layers['cov']` (cytozip's `.X` holds `--score posterior_frac`, an empirical-Bayes shrunken
fraction, so it is not used here). Expected: Pearson r ≈ 1 with tiny residual differences coming from

* blacklist handling — pym3c removes bins overlapping the blacklist by ≥ 20 % of their length (`f=0.2`),
  cytozip drops any feature overlapping a blacklist interval;
* `cov` capping in `.cz` (`uint8` for single cells / `uint16` for pseudobulks, mc scaled with cov so the fraction
  is preserved);
* chrX/chrY/chrM gene exclusion differences for the gene matrices.

In [ ]:
import anndata as ad
import scipy.sparse as sp

VAL_SAMPLE = 'P0'   # any finished sample

def _var_key(adata, var_dim):
    v = adata.var
    cand = [(f'{var_dim}_chrom', f'{var_dim}_start', f'{var_dim}_end'), ('chrom', 'start', 'end')]
    for c, s, e in cand:
        if c in v and s in v and e in v:
            return pd.Index(v[c].astype(str) + ':' + v[s].astype(int).astype(str) + '-' + v[e].astype(int).astype(str))
    return adata.var_names  # gene ids

def compare(sample, feat, mc, base='.'):
    fa = f'{base}/allcools/{sample}/{sample}.{feat}-{mc}.h5ad'; fb = f'{base}/cytozip/{sample}/{sample}.{feat}-{mc}.h5ad'
    if not (os.path.exists(fa) and os.path.exists(fb)):
        print('missing', fa, fb); return None
    A, B = ad.read_h5ad(fa), ad.read_h5ad(fb)
    A.var_names = _var_key(A, feat); B.var_names = _var_key(B, feat)
    cells = A.obs_names.intersection(B.obs_names); feats = A.var_names.intersection(B.var_names)
    A, B = A[cells, feats], B[cells, feats]
    dense = lambda x: (x.toarray() if sp.issparse(x) else np.asarray(x)).astype(np.float64)
    xa = dense(A.X)
    if 'mc' in B.layers and 'cov' in B.layers:
        # raw mc/cov from cytozip's count layers (its .X may hold posterior_frac)
        mcb, cov = dense(B.layers['mc']), dense(B.layers['cov'])
        with np.errstate(invalid='ignore', divide='ignore'):
            xb = np.where(cov > 0, mcb / cov, np.nan)
        m = (cov > 0) & np.isfinite(xa) & np.isfinite(xb)
    else:
        xb = dense(B.X)
        m = np.isfinite(xa) & np.isfinite(xb)
    r = np.corrcoef(xa[m], xb[m])[0, 1]
    diff = np.abs(xa[m] - xb[m])
    res = dict(sample=sample, feature=feat, mc=mc, n_cells_A=A.n_obs, n_cells_B=B.n_obs, n_cells_shared=len(cells),
               n_feat_A=A.n_vars, n_feat_B=B.n_vars, n_feat_shared=len(feats), n_entries=int(m.sum()),
               pearson_r=r, max_abs_diff=diff.max(), median_abs_diff=np.median(diff), frac_diff_gt_0p01=(diff > 0.01).mean())
    return res, xa[m], xb[m]

val_rows, pairs = [], {}
for feat in ['100kb', 'gene']:
    for mc in ['CGN', 'CHN']:
        out = compare(VAL_SAMPLE, feat, mc)
        if out:
            val_rows.append(out[0]); pairs[(feat, mc)] = out[1:]
val = pd.DataFrame(val_rows)
val.to_csv(f'{TABLE_DIR}/validation_{VAL_SAMPLE}.tsv', sep='\t', index=False)
display(val)

if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(1.9 * len(pairs), 2.0), constrained_layout=True)
    for ax, ((feat, mc), (xa, xb)) in zip(np.atleast_1d(axes), pairs.items()):
        rng = np.random.default_rng(0); idx = rng.choice(len(xa), min(200_000, len(xa)), replace=False)
        ax.hexbin(xa[idx], xb[idx], gridsize=60, bins='log', cmap='viridis', lw=0)
        ax.plot([0, 1], [0, 1], 'r--', lw=0.6)
        ax.set_title(f'{feat} m{mc[:2]}  r = {np.corrcoef(xa, xb)[0,1]:.4f}', fontsize=7)
        ax.set_xlabel('ALLCools / pym3c'); ax.set_ylabel('cytozip'); despine(ax)
    savefig(fig, f'fig6_validation_{VAL_SAMPLE}')
    plt.show()

## 10. Numbers for the manuscript text

In [ ]:
if len(full):
    g = full.groupby('pipeline')
    ncell = int(full[full.pipeline == 'cytozip'].n_cells.sum())
    txt = f'''
Across {full.name.nunique()} samples ({ncell:,} cells; {int(full.ncpu.max())} CPUs per job),
building the four feature matrices (100-kb bins and gene bodies, mCG and mCH) took on average
{g.wall_min.mean()['allcools']:.1f} ± {g.wall_min.std()['allcools']:.1f} min per sample with the allc → mcds → h5ad route (ALLCools + pym3c)
versus {g.wall_min.mean()['cytozip']:.1f} ± {g.wall_min.std()['cytozip']:.1f} min with cz → h5ad (cytozip), a
{g.wall_s.sum()['allcools'] / g.wall_s.sum()['cytozip']:.1f}-fold reduction in wall time
({g.cpu_s.sum()['allcools'] / g.cpu_s.sum()['cytozip']:.1f}-fold in CPU time).
Peak memory (proportional set size summed over all worker processes) dropped from
{g.peak_pss_gb.mean()['allcools']:.1f} ± {g.peak_pss_gb.std()['allcools']:.1f} GB to
{g.peak_pss_gb.mean()['cytozip']:.1f} ± {g.peak_pss_gb.std()['cytozip']:.1f} GB
({g.peak_pss_gb.mean()['cytozip'] / g.peak_pss_gb.mean()['allcools'] * 100:.0f} %).'''
    if 'cytozip+allc2cz' in g.groups:
        txt += f'''
Including the one-off allc → cz conversion, cytozip still needed only
{g.wall_min.mean()['cytozip+allc2cz']:.1f} ± {g.wall_min.std()['cytozip+allc2cz']:.1f} min per sample
({g.wall_s.sum()['allcools'] / g.wall_s.sum()['cytozip+allc2cz']:.1f}-fold faster than ALLCools).'''
    print(txt)
    open(f'{TABLE_DIR}/manuscript_numbers.txt', 'w').write(txt)

## 11. Second dataset — human basal-ganglia **cluster pseudobulks** (hg38)

188 cluster-level pseudobulk `allc.tsv.gz` (`~/Projects/BG/pseudobulk/Cluster/allc/`, ~6 GB each, 1.1 TB in total;
mean coverage ~600 per cytosine) — the opposite regime to the mouse data set (many tiny single cells). Same scripts and
commands, selected with `DATASET=bg_cluster` (overrides in `datasets/bg_cluster.sh`):

| setting | mouse single cells | BG pseudobulk | why |
|---|---|---|---|
| reference / indexes | `mm10_with_chrL.{allc,CGN,CHN}.cz` | `hg38_with_chrL.{allc,CGN,CHN}.cz` | genome |
| chrom sizes / gene BED / blacklist | mm10 | `hg38_ucsc.nochrM.sizes`, hg38 `gene_2000_2000.bed.gz`, `hg38-blacklist.v2` | genome |
| `allc2cz -F` | `B,B` (uint8) | `H,H` (uint16) | pseudobulk cov ≫ 255 (max 39 102 in the first 5 M rows) |
| `allc2cz -j` (`A2C_NCPU`) | 96 | 12 | each worker holds one chromosome of tabix lines in memory (~15–18 GB for a dense chr1) |
| `allcools --chunk_size` | 50 | 2 | 188 files / 50 = 4 tasks would leave 92 of 96 CPUs idle; 2 → 94 tasks |
| `NCPU` for both pipelines | 96 | 96 | identical for ALLCools and cytozip |

```shell
cd ~/Projects/test_cytozip/to_adata
# optional: measure the per-worker peak of allc2cz on ONE pseudobulk first, then set A2C_NCPU ≈ (node GB - 20) / peak GB
/usr/bin/time -v ~/Software/conda/m3c/bin/czip allc2cz -I ~/Projects/BG/pseudobulk/Cluster/allc/Oligodendrocyte-c0.allc.tsv.gz \
    -O /tmp/test.cz -r ~/Ref/hg38/hg38_with_chrL.allc.cz -F H,H -C mc,cov -D chrom -u 4,5 2>&1 | grep -E "Maximum resident|Elapsed"
DATASET=bg_cluster bash run_all.sh               # 00 -> 01 (allc -> cz) -> 02 (ALLCools) -> 03 (cytozip)
```

In [ ]:
BG_DIR = os.path.join(WORKDIR, 'bg_cluster')
BG_LOG_DIR, BG_TABLE_DIR = f'{BG_DIR}/logs', f'{BG_DIR}/tables'   # figures go to the shared figures/ dir
os.makedirs(BG_TABLE_DIR, exist_ok=True)
BG_NAME = 'Cluster'

def load_stage_logs(log_dir):
    # last successful row of every (pipeline, name, stage) from bench_run.py TSVs; failed stages are reported
    dfs = []
    for f in sorted(glob.glob(f'{log_dir}/*.tsv')):
        if f.endswith(('.size.tsv', '.cells.tsv')):
            continue
        d = pd.read_csv(f, sep='\t')
        if {'pipeline', 'wall_s', 'exit_code'} <= set(d.columns):
            dfs.append(d.assign(log_file=os.path.basename(f)))
    if not dfs:
        return pd.DataFrame()
    d = pd.concat(dfs, ignore_index=True).sort_values('start_time')
    bad = d[d.exit_code != 0]
    ok = d[d.exit_code == 0].drop_duplicates(['pipeline', 'name', 'stage'], keep='last')
    failed_only = bad.merge(ok[['pipeline', 'name', 'stage']], how='left', indicator=True).query('_merge == "left_only"')
    if len(failed_only):
        print('WARNING — stages without a successful run:'); display(failed_only[['pipeline', 'name', 'stage', 'exit_code', 'log_file']])
    ok = ok.copy()
    ok['cpu_s'] = ok.cpu_user_s + ok.cpu_sys_s
    ok['wall_min'] = ok.wall_s / 60
    ok['is_subset'] = ok['name'].astype(str).str.contains('_sub')
    return ok

bg_log = load_stage_logs(BG_LOG_DIR)
if len(bg_log):
    bg_log.to_csv(f'{BG_TABLE_DIR}/all_stages.tsv', sep='\t', index=False)
    stage_tbl = bg_log.assign(cpu_h=bg_log.cpu_s / 3600, peak_pss_gb=bg_log.max_pss_sum_mb / 1024,
                              peak_rss_gb=bg_log.max_rss_sum_mb / 1024, output_gb=bg_log.output_size_mb / 1024)
    display(stage_tbl[['pipeline', 'stage', 'n_cells', 'ncpu', 'wall_min', 'cpu_h', 'peak_pss_gb', 'peak_rss_gb', 'output_gb']].round(2))
else:
    print(f'no BG logs yet in {BG_LOG_DIR} — run: DATASET=bg_cluster bash run_all.sh')

In [ ]:
# per-pipeline totals (sum over stages; peak memory = max over stages), + one-off allc -> cz
bg_sum = pd.DataFrame()
if len(bg_log):
    main_bg = bg_log[bg_log.pipeline.isin(['allcools', 'cytozip'])]
    rows = []
    for p, d in main_bg.groupby('pipeline'):
        rows.append(dict(pipeline=p, n_stages=len(d), n_cells=d.n_cells.max(), ncpu=d.ncpu.max(),
                         wall_s=d.wall_s.sum(), cpu_s=d.cpu_s.sum(),
                         peak_pss_mb=d.max_pss_sum_mb.max(), peak_rss_mb=d.max_rss_sum_mb.max(),
                         h5ad_mb=d.loc[d.stage.str.startswith('h5ad'), 'output_size_mb'].sum(),
                         mcds_mb=d.loc[d.stage.str.startswith('mcds'), 'output_size_mb'].sum()))
    a2c_bg = bg_log[bg_log.pipeline == 'allc2cz']
    if len(a2c_bg) and 'cytozip' in main_bg.pipeline.values:
        cz = [r for r in rows if r['pipeline'] == 'cytozip'][0]
        a = a2c_bg.iloc[0]
        rows.append(dict(cz, pipeline='cytozip+allc2cz', wall_s=cz['wall_s'] + a.wall_s, cpu_s=cz['cpu_s'] + a.cpu_s,
                         peak_pss_mb=max(cz['peak_pss_mb'], a.max_pss_sum_mb), peak_rss_mb=max(cz['peak_rss_mb'], a.max_rss_sum_mb)))
    bg_sum = pd.DataFrame(rows).set_index('pipeline')
    bg_sum['wall_h'] = bg_sum.wall_s / 3600; bg_sum['cpu_h'] = bg_sum.cpu_s / 3600
    bg_sum['peak_pss_gb'] = bg_sum.peak_pss_mb / 1024; bg_sum['peak_rss_gb'] = bg_sum.peak_rss_mb / 1024
    bg_sum.to_csv(f'{BG_TABLE_DIR}/summary.tsv', sep='\t')
    display(bg_sum[['n_cells', 'ncpu', 'n_stages', 'wall_h', 'cpu_h', 'peak_pss_gb', 'peak_rss_gb', 'h5ad_mb', 'mcds_mb']].round(2))
    if {'allcools', 'cytozip'} <= set(bg_sum.index):
        A, C = bg_sum.loc['allcools'], bg_sum.loc['cytozip']
        print(f"BG pseudobulk: wall speed-up {A.wall_s / C.wall_s:.2f}x, CPU speed-up {A.cpu_s / C.cpu_s:.2f}x, "
              f"peak PSS {C.peak_pss_gb:.1f} vs {A.peak_pss_gb:.1f} GB ({C.peak_pss_gb / A.peak_pss_gb * 100:.0f}%)")
        if 'cytozip+allc2cz' in bg_sum.index:
            print(f"   incl. allc2cz: {A.wall_s / bg_sum.loc['cytozip+allc2cz', 'wall_s']:.2f}x faster (wall)")

# allc -> cz cost and storage
bg_size = None
f_size = f'{BG_LOG_DIR}/allc2cz.{BG_NAME}.size.tsv'
if os.path.exists(f_size):
    bg_size = pd.read_csv(f_size, sep='\t').iloc[0]
    a = bg_log[bg_log.pipeline == 'allc2cz'].iloc[0]
    print(f"allc2cz: {int(bg_size.n_allc)} pseudobulks, {a.wall_s / 3600:.2f} h wall, {a.cpu_s / 3600:.1f} CPU-h on {int(a.ncpu)} workers, "
          f"peak PSS {a.max_pss_sum_mb / 1024:.1f} GB (largest process {a.max_rss_single_mb / 1024:.1f} GB); "
          f"{bg_size.allc_dir_mb / 1024:.1f} GB allc -> {bg_size.cz_dir_mb / 1024:.1f} GB cz "
          f"({bg_size.cz_dir_mb / bg_size.allc_dir_mb * 100:.1f}%)")

In [ ]:
# Figure 7 — BG pseudobulk: stacked runtime by stage, CPU time, peak memory, storage
if len(bg_sum) and {'allcools', 'cytozip'} <= set(bg_sum.index):
    st_a = ['mcds-100kb', 'mcds-gene', 'h5ad-100kb-CGN+CHN', 'h5ad-gene-CGN', 'h5ad-gene-CHN']
    st_b = ['h5ad-100kb-CGN', 'h5ad-100kb-CHN', 'h5ad-gene-CGN', 'h5ad-gene-CHN']
    wall = bg_log.set_index(['pipeline', 'stage']).wall_s / 3600
    has_a2c = 'cytozip+allc2cz' in bg_sum.index
    fig, axes = plt.subplots(1, 4, figsize=(7.4, 2.5), constrained_layout=True,
                             gridspec_kw=dict(width_ratios=[1.5, 1, 1, 1]))
    ax = axes[0]
    bars = [('allcools', st_a, plt.get_cmap('Blues')(np.linspace(0.45, 0.9, len(st_a))), 0),
            ('cytozip', st_b, plt.get_cmap('Reds')(np.linspace(0.45, 0.9, len(st_b))), 1)]
    for p, stages, cm, xi in bars:
        bottom = 0
        for st, c in zip(stages, cm):
            v = wall.get((p, st), np.nan)
            if np.isnan(v):
                continue
            ax.bar(xi, v, bottom=bottom, color=c, width=0.65, edgecolor='white', lw=0.4, label=f'{LABEL[p]}: {st}')
            bottom += v
        if p == 'cytozip' and has_a2c:   # one-off conversion stacked (hatched) on top of cytozip
            v = wall.get(('allc2cz', 'allc2cz'), np.nan)
            ax.bar(xi, v, bottom=bottom, color='white', edgecolor=COLORS['cytozip'], hatch='////', lw=0.6, width=0.65,
                   label=f"allc→cz ({int(bg_log[bg_log.pipeline == 'allc2cz'].ncpu.iloc[0])} CPUs, one-off)")
            ax.text(xi + 0.36, bottom, f'{bottom:.1f} h', ha='left', va='center', fontsize=7)   # cz->h5ad only
            ax.text(xi, bottom + v, f'{bottom + v:.1f} h\n(+allc→cz)', ha='center', va='bottom', fontsize=6, color='grey')
        else:
            ax.text(xi, bottom, f'{bottom:.1f} h', ha='center', va='bottom', fontsize=7)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['ALLCools', 'cytozip'], fontsize=7); ax.set_xlim(-0.6, 1.9)
    ax.set_ylabel('Wall time (h)'); ax.set_title('Runtime by stage', fontsize=8); despine(ax)
    ax.margins(y=0.15)
    fig.legend(*ax.get_legend_handles_labels(), loc='lower center', bbox_to_anchor=(0.5, -0.16), ncol=4, fontsize=5.5, frameon=False)
    order_bg = ['allcools', 'cytozip'] + (['cytozip+allc2cz'] if has_a2c else [])
    for ax, (col, ylabel, title) in zip(axes[1:3], [('cpu_h', 'CPU time (CPU-h)', 'CPU time'),
                                                     ('peak_pss_gb', 'Peak memory (GB, Σ PSS)', 'Peak memory')]):
        v = bg_sum.loc[order_bg, col]
        ax.bar(range(len(order_bg)), v, color=[cols[p] for p in order_bg], width=0.65)
        for i, x in enumerate(v):
            ax.text(i, x, f'{x:.1f}', ha='center', va='bottom', fontsize=6.5)
        ax.set_xticks(range(len(order_bg))); ax.set_xticklabels([labels[p] for p in order_bg], fontsize=6.5)
        ax.set_ylabel(ylabel); ax.set_title(title, fontsize=8); despine(ax)
    ax = axes[3]
    if bg_size is not None:
        vals = [bg_size.allc_dir_mb / 1024, bg_size.cz_dir_mb / 1024]
        ax.bar([0, 1], vals, color=['#7f7f7f', COLORS['cytozip']], width=0.65)
        for i, x in enumerate(vals):
            ax.text(i, x, f'{x:.0f}', ha='center', va='bottom', fontsize=6.5)
        ax.set_xticks([0, 1]); ax.set_xticklabels(['allc.tsv.gz', '.cz'], fontsize=7)
        ax.set_ylabel('Total size (GB)'); ax.set_title(f'Storage ({vals[1] / vals[0] * 100:.0f}%)', fontsize=8); despine(ax)
    else:
        ax.axis('off')
    fig.suptitle(f"{int(bg_sum.n_cells.max())} human BG cluster pseudobulks (hg38), {int(bg_sum.ncpu.max())} CPUs per pipeline", fontsize=8)
    savefig(fig, 'fig7_bg_pseudobulk')
    plt.show()
else:
    print('BG results incomplete — figure skipped')

In [ ]:
# Figure 8 — both datasets side by side: fold change of cytozip vs ALLCools
fold_rows = []
if len(full) and {'allcools', 'cytozip'} <= set(full.pipeline):
    t = full.groupby('pipeline')[['wall_s', 'cpu_s']].sum(); m = full.groupby('pipeline').peak_pss_gb.mean()
    fold_rows.append(dict(dataset=f'Mouse\nsingle cells\n(n={int(full[full.pipeline == "cytozip"].n_cells.sum()):,})',
                          wall=t.loc['allcools', 'wall_s'] / t.loc['cytozip', 'wall_s'],
                          wall_a2c=(t.loc['allcools', 'wall_s'] / t.loc['cytozip+allc2cz', 'wall_s']) if 'cytozip+allc2cz' in t.index else np.nan,
                          cpu=t.loc['allcools', 'cpu_s'] / t.loc['cytozip', 'cpu_s'],
                          mem=m.loc['cytozip'] / m.loc['allcools']))
if len(bg_sum) and {'allcools', 'cytozip'} <= set(bg_sum.index):
    A, C = bg_sum.loc['allcools'], bg_sum.loc['cytozip']
    fold_rows.append(dict(dataset=f'Human BG\npseudobulk\n(n={int(C.n_cells)})',
                          wall=A.wall_s / C.wall_s,
                          wall_a2c=(A.wall_s / bg_sum.loc['cytozip+allc2cz', 'wall_s']) if 'cytozip+allc2cz' in bg_sum.index else np.nan,
                          cpu=A.cpu_s / C.cpu_s, mem=C.peak_pss_gb / A.peak_pss_gb))
fold = pd.DataFrame(fold_rows)
if len(fold):
    fold.assign(dataset=fold.dataset.str.replace('\n', ' ')).to_csv(f'{TABLE_DIR}/fold_change_by_dataset.tsv', sep='\t', index=False)
    display(fold.round(2))
    fig, axes = plt.subplots(1, 3, figsize=(6.6, 2.4), constrained_layout=True)
    x = np.arange(len(fold)); w = 0.38
    ax = axes[0]
    ax.bar(x - w/2, fold.wall, w, color=COLORS['cytozip'], label='cz→h5ad')
    ax.bar(x + w/2, fold.wall_a2c, w, color=COLORS['cytozip (+allc2cz)'], label='incl. allc→cz')
    ax.set_ylabel('Speed-up (ALLCools / cytozip)'); ax.set_title('Wall time', fontsize=8); ax.legend(frameon=False, fontsize=6)
    axes[1].bar(x, fold.cpu, 0.6, color=COLORS['cytozip']); axes[1].set_ylabel('Speed-up (ALLCools / cytozip)'); axes[1].set_title('CPU time', fontsize=8)
    axes[2].bar(x, fold.mem * 100, 0.6, color=COLORS['cytozip']); axes[2].set_ylabel('cytozip peak memory\n(% of ALLCools)'); axes[2].set_title('Peak memory (Σ PSS)', fontsize=8)
    for ax, col, off, fmt in [(axes[0], 'wall', -w/2, '{:.1f}×'), (axes[0], 'wall_a2c', w/2, '{:.1f}×'),
                              (axes[1], 'cpu', 0, '{:.1f}×'), (axes[2], 'mem', 0, '{:.0f}%')]:
        vals = fold[col] * (100 if col == 'mem' else 1)
        for i, v in enumerate(vals):
            if np.isfinite(v):
                ax.text(i + off, v, fmt.format(v), ha='center', va='bottom', fontsize=6.5)
    for ax, ref in [(axes[0], 1), (axes[1], 1), (axes[2], 100)]:
        ax.axhline(ref, color='grey', lw=0.6, ls='--')
        ax.set_xticks(x); ax.set_xticklabels(fold.dataset, fontsize=6.5); ax.margins(y=0.12); despine(ax)
    savefig(fig, 'fig8_fold_change_by_dataset')
    plt.show()
else:
    print('need finished results of at least one dataset')

In [ ]:
# validation on the BG pseudobulks (same comparison as section 9)
bg_val_rows, bg_pairs = [], {}
for feat in ['100kb', 'gene']:
    for mc in ['CGN', 'CHN']:
        out = compare(BG_NAME, feat, mc, base=BG_DIR)
        if out:
            bg_val_rows.append(out[0]); bg_pairs[(feat, mc)] = out[1:]
if bg_val_rows:
    bg_val = pd.DataFrame(bg_val_rows)
    bg_val.to_csv(f'{BG_TABLE_DIR}/validation_{BG_NAME}.tsv', sep='\t', index=False)
    display(bg_val)
    fig, axes = plt.subplots(1, len(bg_pairs), figsize=(1.9 * len(bg_pairs), 2.0), constrained_layout=True)
    for ax, ((feat, mc), (xa, xb)) in zip(np.atleast_1d(axes), bg_pairs.items()):
        idx = np.random.default_rng(0).choice(len(xa), min(200_000, len(xa)), replace=False)
        ax.hexbin(xa[idx], xb[idx], gridsize=60, bins='log', cmap='viridis', lw=0)
        ax.plot([0, 1], [0, 1], 'r--', lw=0.6)
        ax.set_title(f'BG {feat} m{mc[:2]}  r = {np.corrcoef(xa, xb)[0,1]:.4f}', fontsize=7)
        ax.set_xlabel('ALLCools / pym3c'); ax.set_ylabel('cytozip'); despine(ax)
    savefig(fig, 'fig9_validation_bg_pseudobulk')
    plt.show()

In [ ]:
if len(bg_sum) and {'allcools', 'cytozip'} <= set(bg_sum.index):
    A, C = bg_sum.loc['allcools'], bg_sum.loc['cytozip']
    txt = f'''
On {int(C.n_cells)} human basal-ganglia cluster pseudobulks (hg38; {bg_size.allc_dir_mb / 1024 / 1024 if bg_size is not None else float("nan"):.1f} TB of allc),
building the same four matrices took {A.wall_s / 3600:.1f} h with ALLCools + pym3c and {C.wall_s / 3600:.1f} h with cytozip
({A.wall_s / C.wall_s:.1f}-fold faster; {A.cpu_s / C.cpu_s:.1f}-fold less CPU time) on {int(C.ncpu)} CPUs, with a peak memory of
{C.peak_pss_gb:.1f} GB versus {A.peak_pss_gb:.1f} GB.'''
    if bg_size is not None:
        a = bg_log[bg_log.pipeline == 'allc2cz'].iloc[0]
        txt += f'''
The one-off conversion to .cz took {a.wall_s / 3600:.1f} h on {int(a.ncpu)} workers and reduced storage from
{bg_size.allc_dir_mb / 1024:.0f} GB to {bg_size.cz_dir_mb / 1024:.0f} GB ({bg_size.cz_dir_mb / bg_size.allc_dir_mb * 100:.0f}%).'''
    print(txt)
    open(f'{BG_TABLE_DIR}/manuscript_numbers.txt', 'w').write(txt)